[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jobellet/paper2audio/blob/main/notebooks/paper2audio_colab.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/jobellet/paper2audio/blob/main/notebooks/paper2audio_colab.ipynb)

# paper2audio on a free GPU

Turn a scientific paper (PDF) into an MP3 with one chapter per section. The text is
cleaned (citations, statistics, figure references, running heads and captions
removed) and read in the right column order by
[Kokoro-82M](https://huggingface.co/hexgrad/Kokoro-82M), a natural-sounding neural voice.

**Colab:** Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all, and
pick your PDF when the upload button appears.

**Kaggle:** in the right panel, Settings → Accelerator → **GPU T4** and Internet → **on**.
Add your PDF with **Add Input → Upload** (or paste a link in `PDF_URL`), then Run All.
The MP3 appears under Output.

Without a GPU it still works, only more slowly.

In [ ]:
#@title 1. Install (about 1 minute)
import os, shutil, subprocess, sys

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
GPU = shutil.which("nvidia-smi") is not None and subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0


def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)


pip("paper2audio[kokoro] @ git+https://github.com/jobellet/paper2audio")
if GPU:
    # kokoro-onnx installs the CPU build of onnxruntime; the GPU build runs Kokoro on
    # CUDA. Both share the module's files, so the GPU build is reinstalled every time.
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-q", "-y", "onnxruntime"], check=False)
    pip("--force-reinstall", "--no-deps", "onnxruntime-gpu")
if not shutil.which("ffmpeg"):
    subprocess.run("apt-get -qq -y install ffmpeg > /dev/null", shell=True, check=False)

import onnxruntime
print("Python", sys.version.split()[0], "| GPU:", GPU, "| onnxruntime", onnxruntime.__version__,
      onnxruntime.get_available_providers())

In [ ]:
#@title 2. Settings
VOICE = "af_heart"  #@param ["af_heart", "af_bella", "af_nicole", "am_michael", "am_fenrir", "am_puck", "bf_emma", "bf_isabella", "bm_george", "bm_fable"]
SPEED = 1.1  #@param {type:"slider", min:0.7, max:1.6, step:0.05}
READ_CAPTIONS = False  #@param {type:"boolean"}
SKIP_SECTIONS = ""  #@param {type:"string"}
#@markdown `SKIP_SECTIONS`: comma-separated regexes, e.g. `methods, materials`.
PREVIEW = False  #@param {type:"boolean"}
#@markdown `PREVIEW`: only read the title, the abstract and the first section.
PDF_URL = ""  #@param {type:"string"}
#@markdown `PDF_URL`: a direct link to a PDF, instead of uploading one.

In [ ]:
#@title 3. Choose the paper
import glob
import urllib.request
from pathlib import Path

work = Path("/kaggle/working") if IN_KAGGLE else Path.cwd()
if PDF_URL.strip():
    name = Path(PDF_URL.split("?")[0]).name or "paper"
    pdf = work / (name if name.lower().endswith(".pdf") else name + ".pdf")
    request = urllib.request.Request(PDF_URL.strip(), headers={"User-Agent": "Mozilla/5.0"})
    pdf.write_bytes(urllib.request.urlopen(request).read())
elif IN_COLAB:
    from google.colab import files
    uploaded = files.upload()
    pdf = next(Path(name) for name in uploaded if name.lower().endswith(".pdf"))
else:
    found = sorted(glob.glob("/kaggle/input/**/*.pdf", recursive=True)) + sorted(glob.glob("*.pdf"))
    if not found:
        raise SystemExit("No PDF found: paste a link in PDF_URL, or on Kaggle add your PDF with Add Input → Upload.")
    pdf = Path(found[0])
print("paper:", pdf)

In [ ]:
#@title 4. Extract and clean the text
from paper2audio import run

options = {}
if READ_CAPTIONS:
    options["read_captions"] = True
if SKIP_SECTIONS.strip():
    options["skip_sections"] = tuple(s.strip() for s in SKIP_SECTIONS.split(",") if s.strip())
script = run(str(pdf), **options)
if PREVIEW:
    script.sections = script.sections[:2]

transcript = work / f"{pdf.stem}.txt"
transcript.write_text(script.to_text(), encoding="utf-8")
print(f"profile: {script.profile}, year: {script.year or 'unknown'}")
print("sections:", " | ".join(s.title or "(untitled)" for s in script.sections))
if script.skipped:
    print("skipped: ", " | ".join(script.skipped))
print(f"{len(script.to_text().split())} words; transcript saved to {transcript}")

In [ ]:
#@title 5. Read it aloud and download
import time

import onnxruntime
from IPython.display import Audio, FileLink, display
from paper2audio.audio import render
from paper2audio.tts import get_backend

if GPU and hasattr(onnxruntime, "preload_dlls"):
    onnxruntime.preload_dlls()  # CUDA and cuDNN libraries from the nvidia-* pip packages

backend = get_backend("kokoro", voice=VOICE, rate=SPEED)
start = time.time()
mp3 = render(script, backend, work / f"{pdf.stem}.mp3")
device = backend._kokoro().sess.get_providers()[0].replace("ExecutionProvider", "")
print(f"done in {time.time() - start:.0f} s on {device}: {mp3}")
display(Audio(str(mp3)))
if IN_COLAB:
    files.download(str(mp3))
    files.download(str(transcript))
else:
    display(FileLink(os.path.relpath(mp3)), FileLink(os.path.relpath(transcript)))